# Sri Lankan Food Allergen Risk Prediction — ML Model

This notebook trains a **Multi-Output Random Forest Classifier** to predict a user's risk level (0–3) for each of 31 food allergens based on their demographic and medical profile.

| Risk Level | Meaning |
|---|---|
| 0 | No reaction |
| 1 | Mild Intolerance (Stomach pain, Nausea) |
| 2 | Mild Allergic Reaction (Skin rash, Itching, Runny nose) |
| 3 | Severe Allergic Reaction (Swelling, Breathing difficulty) |

In [8]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from sklearn.model_selection import train_test_split
from xgboost import XGBClassifier
from sklearn.metrics import classification_report
from imblearn.pipeline import Pipeline
from imblearn.over_sampling import SMOTE

sns.set_theme(style="whitegrid", palette="muted")
print("Libraries loaded successfully.")

Libraries loaded successfully.


## 0. Load Preprocessed Data

In [9]:
# In a Jupyter notebook, __file__ is not available.
# Path is resolved relative to the notebook location (ml_model/scripts/).
DATA_PATH = Path('..') / 'data' / 'processed' / 'preprocessed_data.csv'
ARTIFACTS_DIR = Path('..') / 'artifacts'
ARTIFACTS_DIR.mkdir(exist_ok=True)

df = pd.read_csv(DATA_PATH)
print(f"Loaded dataset: {df.shape[0]} rows x {df.shape[1]} columns")
df.head()

Loaded dataset: 90 rows x 83 columns


,age,gender,allergy_awareness,personal_allergy_history,outside_food_frequency,lactose_intolerance,milk,eggs,peanuts,tree_nuts,...,tomato,rambutan,sarana,avocado,breadfruit,calms,gotukola,mustard,fenugreek,cumin
0,0.088235,1,1,1,1,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1,0.161765,0,1,0,1,1,1,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2,0.147059,0,1,0,1,0,0,0,0,1,...,0,0,0,0,0,0,0,0,0,0
3,0.161765,0,1,1,1,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
4,0.250000,1,1,0,1,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


## 1. Define Features (X) and Targets (y)

- **X** — User's demographic & medical profile  
- **y** — Risk level (0–3) for each of the 31 allergen columns

### Target columns
- **PART 1** — 17 standard grid foods (0/1/2/3 scale from `risk_mapping`)
- **PART 2** — 14 other identified foods (0 or 2, extracted from free-text columns)

In [10]:
# ── Target columns ────────────────────────────────────────────────────────────

# PART 1: Standard grid food columns (0-3 scale from risk_mapping)
grid_food_cols = [
    'milk', 'eggs', 'peanuts', 'tree_nuts', 'wheat', 'soy', 'sesame', 'prawns',
    'crab', 'cuttlefish / squid', 'dry_fish', 'tuna', 'beef', 'pork', 'mutton', 'coconut',
    'spicy_oily'
]

# PART 2: Other identified food columns (0 or 2, extracted from free-text columns)
other_food_cols = [
    'jackfruit', 'cassava', 'moringa', 'pineapple', 'tomato', 'rambutan',
    'sarana', 'avocado', 'breadfruit', 'calms', 'gotukola',
    'mustard', 'fenugreek', 'cumin'
]

# Combined list of all allergen targets the model will predict
identified_allergens = grid_food_cols + other_food_cols

# ── Feature columns (X) ───────────────────────────────────────────────────────
feature_cols = [
    'age', 'gender','allergy_awareness', 'personal_allergy_history',
    'outside_food_frequency', 'lactose_intolerance',
    'med_cond_asthma', 'med_cond_eczema', 'med_cond_food_allergy',
    'med_cond_none_of_these', 'med_cond_not_sure',
    # Aggregated family history flag
    'family_has_history',
    # Specific family history allergen flags
    'family_asthma', 'family_eczema',
    'family_allergy_beef', 'family_allergy_pork', 'family_allergy_sausage',
    'family_allergy_red_meat', 'family_allergy_seafood', 'family_allergy_dairy',
    'family_allergy_nuts', 'family_allergy_egg', 'family_allergy_tomato',
    'family_allergy_pineapple', 'family_allergy_avocado',
    'family_allergy_breadfruit', 'family_allergy_flour',
]
# One-hot encoded columns added dynamically
feature_cols += [col for col in df.columns if col.startswith('province_')]
feature_cols += [col for col in df.columns if col.startswith('blood_type_')]
feature_cols += [col for col in df.columns if col.startswith('dietary_pattern_')]
feature_cols += [col for col in df.columns if col.startswith('work_env_')]

# ── Validate ──────────────────────────────────────────────────────────────────
missing_features = [c for c in feature_cols if c not in df.columns]
missing_targets  = [c for c in identified_allergens if c not in df.columns]

if missing_features:
    raise ValueError(f"Missing feature columns in dataset: {missing_features}")
if missing_targets:
    raise ValueError(f"Missing target columns in dataset: {missing_targets}")


In [11]:
import pandas as pd
import numpy as np

# 1. Calculate the full Spearman correlation matrix
full_corr = df[feature_cols + identified_allergens].corr(method='spearman')

# 2. Isolate Features (Rows) vs Targets (Columns)
feature_target_corr = full_corr.loc[feature_cols, identified_allergens]

# 3. Find the MAXIMUM absolute correlation for each feature across ALL 31 targets
# (If a feature is useless for every single food, its max score will be very close to 0)
max_correlations = feature_target_corr.abs().max(axis=1)

# 4. Set your Noise Threshold
# 0.15 is a standard starting point for survey data. 
# Anything below 0.15 is generally considered statistical noise.
THRESHOLD = 0.15

# 5. Automatically separate the "Keepers" from the "Trash"
features_to_keep = max_correlations[max_correlations >= THRESHOLD].index.tolist()
features_to_drop = max_correlations[max_correlations < THRESHOLD].index.tolist()

# 6. Create your brand new, lean X matrix!
X_optimized = df[features_to_keep]

print("--- FEATURE OPTIMIZATION RESULTS ---")
print(f"Original Feature Count : {len(feature_cols)}")
print(f"Optimized Feature Count: {len(features_to_keep)}")
print(f"Dropped Columns        : {len(features_to_drop)}")
print("-" * 36)



X = df[features_to_keep]
y = df[identified_allergens]

print(f"Feature matrix X : {X.shape[0]} rows  x  {X.shape[1]} features")
print(f"Target matrix  y : {y.shape[0]} rows  x  {y.shape[1]} allergen targets")
print(f"\nAllergen targets : {identified_allergens}")

--- FEATURE OPTIMIZATION RESULTS ---
Original Feature Count : 52
Optimized Feature Count: 44
Dropped Columns        : 5
------------------------------------
Feature matrix X : 90 rows  x  44 features
Target matrix  y : 90 rows  x  31 allergen targets

Allergen targets : ['milk', 'eggs', 'peanuts', 'tree_nuts', 'wheat', 'soy', 'sesame', 'prawns', 'crab', 'cuttlefish / squid', 'dry_fish', 'tuna', 'beef', 'pork', 'mutton', 'coconut', 'spicy_oily', 'jackfruit', 'cassava', 'moringa', 'pineapple', 'tomato', 'rambutan', 'sarana', 'avocado', 'breadfruit', 'calms', 'gotukola', 'mustard', 'fenugreek', 'cumin']


## 3. Train the Multi-Output Model


In [12]:
# This dictionary will store all 31 of your trained AI models!
trained_risk_engines = {}

print("🚀 STARTING MASS TRAINING FOR ALL 31 ALLERGENS...\n")

for target_food in identified_allergens:
    # 1. Binarize the target
    y_single_target = y[target_food].apply(lambda x: 1 if x > 0 else 0)
    positive_cases = y_single_target.sum()
    
    # Skip foods where absolutely no one had an allergy (can't train an AI on 0 cases)
    if positive_cases < 2:
        print(f"⏭️ Skipping {target_food.upper()}: 0 positive cases in dataset.")
        continue
        
    # 2. Train-Test Split for this specific food (Using Pandas directly)
    X_train, X_test, y_train, y_test = train_test_split(
        X, y_single_target, 
        test_size=0.2, random_state=42, stratify=y_single_target
    )
    
    # 3. Dynamic Pipeline Creation
    if positive_cases >= 4:
        # If we have enough data, use the powerful SMOTE pipeline
        model = Pipeline([
            ('smote', SMOTE(random_state=42, k_neighbors=2)),
            ('xgb', XGBClassifier(
                n_estimators=100, max_depth=3, learning_rate=0.05, 
                eval_metric='logloss', random_state=42
            ))
        ])
    else:
        # If data is too rare for SMOTE, fallback to standard XGBoost with weighted classes
        # scale_pos_weight forces the AI to pay attention to the rare allergy
        weight = (len(y_train) - positive_cases) / positive_cases
        model = XGBClassifier(
            n_estimators=100, max_depth=3, learning_rate=0.05, 
            scale_pos_weight=weight, eval_metric='logloss', random_state=42
        )
    
    # 4. Train the model and save it to our dictionary
    try:
        model.fit(X_train, y_train)
        trained_risk_engines[target_food] = model
        print(f"✅ Successfully trained model for: {target_food.upper()} (Cases: {positive_cases})")
    except Exception as e:
        print(f"❌ Failed to train {target_food.upper()}: {e}")

print("\n🎉 ALL MODELS TRAINED SUCCESSFULLY!")
print(f"Total Active Models in Memory: {len(trained_risk_engines)}")

🚀 STARTING MASS TRAINING FOR ALL 31 ALLERGENS...

✅ Successfully trained model for: MILK (Cases: 13)
✅ Successfully trained model for: EGGS (Cases: 3)
✅ Successfully trained model for: PEANUTS (Cases: 5)
✅ Successfully trained model for: TREE_NUTS (Cases: 6)
✅ Successfully trained model for: WHEAT (Cases: 5)
✅ Successfully trained model for: SOY (Cases: 5)
✅ Successfully trained model for: SESAME (Cases: 5)
✅ Successfully trained model for: PRAWNS (Cases: 17)
✅ Successfully trained model for: CRAB (Cases: 18)
✅ Successfully trained model for: CUTTLEFISH / SQUID (Cases: 18)
✅ Successfully trained model for: DRY_FISH (Cases: 4)
✅ Successfully trained model for: TUNA (Cases: 11)
✅ Successfully trained model for: BEEF (Cases: 27)
✅ Successfully trained model for: PORK (Cases: 31)
✅ Successfully trained model for: MUTTON (Cases: 18)
✅ Successfully trained model for: COCONUT (Cases: 2)
✅ Successfully trained model for: SPICY_OILY (Cases: 18)
✅ Successfully trained model for: JACKFRUIT (Cases

In [16]:
# ==========================================
# SIMULATED BACKEND: SCANNING A RECIPE
# ==========================================
def scan_recipe_for_user(user_profile_df, detected_ingredients, all_models):
    """
    Simulates the backend scanning a specific recipe for a specific user.
    """
    print(f"--- 🔍 SCANNING DISH ---")
    print(f"Detected Ingredients: {detected_ingredients}\n")
    
    alerts = []
    # Convert user profile to numpy to avoid the bug
    user_np = user_profile_df.astype('float32').to_numpy()
    
    # Only run the ML models for the ingredients actually IN the dish
    for ingredient in detected_ingredients:
        clean_name = ingredient.lower().replace(' ', '_')
        
        # Check if we have a trained model for this ingredient
        if clean_name in all_models:
            model = all_models[clean_name]
            
            # Predict 0 (Safe) or 1 (Risk)
            prediction = model.predict(user_np)[0]
            probability = model.predict_proba(user_np)[0][1] * 100
            
            if prediction == 1:
                alerts.append(f"🚨 RISK: {probability:.1f}% chance of reacting to {ingredient.title()}!")
            elif probability > 20.0: # Optional: Warn them if probability is elevated but not severe
                alerts.append(f"⚠️ CAUTION: {probability:.1f}% chance of reacting to {ingredient.title()}.")
                
    if not alerts:
        return "✅ Safe to eat! No risks detected for your profile."
    
    return "\n".join(alerts)

# --- LET'S TEST IT! ---
# Create our "High Risk" Kamal again (Asthma + Family Seafood Allergy)
test_user = pd.DataFrame(0, index=[0], columns=X.columns)
test_user['age'] = 25 
test_user['gender'] = 0 
test_user['med_cond_asthma'] = 1
test_user['family_has_history'] = 1
test_user['family_allergy_seafood'] = 1
if 'province_Western Province' in X.columns: test_user['province_Western Province'] = 1
if 'blood_type_O+' in X.columns:
    test_user['blood_type_A+'] = 1

# Simulate the RAG system identifying a complex dish
dish_ingredients = ['prawns', 'cuttlefish / squid', 'coconut', 'garlic']

# Run the scan!
print(scan_recipe_for_user(test_user, dish_ingredients, trained_risk_engines))

--- 🔍 SCANNING DISH ---
Detected Ingredients: ['prawns', 'cuttlefish / squid', 'coconut', 'garlic']

🚨 RISK: 73.0% chance of reacting to Prawns!
